# 02 — Reference Methods from the ICD Coding Survey

<details open>
<summary><strong>Why This Notebook Exists</strong></summary>

After the EDA and preprocessing phases, we paused before implementing more models. The reason is simple: if we only try methods because they are available in scikit-learn or Transformers, the project becomes a leaderboard chase. Instead, we want our model choices to be grounded in the ICD coding literature.

The main reference for this notebook is Yan et al. (2022), *A survey of automated International Classification of Diseases coding: development, challenges, and applications*. After reading the survey, we understood that ICD coding has its own structure, risks, and history. This helped us decide which ideas are feasible for our Kaggle assignment and which ones should stay as future work.
</details>

In [ ]:
from pathlib import Path
import sys
import subprocess
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

TABLES = PROJECT_ROOT / 'reports' / 'tables'
FIGURES = PROJECT_ROOT / 'reports' / 'figures'

## <details open><summary>1. Why Automated ICD Coding Matters</summary></details>

Yan et al. describe ICD coding as a central process in healthcare information systems. Manual coding is slow because trained coders must read clinical documentation and assign standardized codes. Mistakes are not just academic: coding errors can affect reimbursement, hospital management, statistics, diagnosis-related groups (DRGs), and medical record management.

This helped us understand that even our simplified Kaggle task is connected to a real workflow. We are not predicting labels for an abstract benchmark; we are learning how clinical language can be transformed into standardized categories.

## <details open><summary>2. Why ICD Coding Is Not Ordinary Text Classification</summary></details>

After reading the survey, we understood that ICD codes are not independent flat labels. ICD has a hierarchy: broad chapters, categories, and more specific descendants. The survey highlights relationships such as:

- **Parent-child inheritance:** a child code is a more specific version of a parent concept.
- **Sibling mutual exclusion:** sibling codes may represent alternatives along the same clinical axis.
- **Friend-node co-occurrence:** codes from different branches can appear together because diseases, procedures, and comorbidities interact.
- **Medical terminology:** the text contains abbreviations, synonyms, and specialized clinical expressions.

This helped us decide not to treat the problem as generic sentiment classification or topic classification. Even though our final target is only the first character of `Code`, the underlying data still comes from a structured medical coding system.

## <details open><summary>3. Historical Method Evolution</summary></details>

Yan et al. organize automated ICD coding as a progression from hand-written systems to representation-learning methods. We summarize that evolution here and then position our project at the end: a scoped category-prefix task over short literals.

In [ ]:
subprocess.run([sys.executable, str(PROJECT_ROOT / 'scripts' / 'create_survey_method_map.py')], check=True)
display(Image(filename=FIGURES / 'fig_09_method_evolution_timeline.png'))

The evolution is useful because it prevents us from pretending that RoBERTa is the only reasonable option. Rule-based systems are interpretable but brittle. Traditional ML is still strong for short texts. CNN/RNN methods introduced neural encoders. GNN and knowledge-based methods exploit the ICD structure. PLM/Transformer methods bring strong pretrained representations, which is why we use a Spanish biomedical-clinical RoBERTa backbone.

## <details open><summary>4. Four Challenges from the Survey</summary></details>

The survey helped us focus on four recurring challenges:

1. **Large label space.** Real ICD coding can involve tens of thousands of possible codes.
2. **Unbalanced label distribution.** Common conditions dominate; rare codes have very few examples.
3. **Long document text.** Many ICD coding datasets use full EMRs or discharge summaries, which can exceed Transformer input limits.
4. **Interpretability.** Clinical systems need explanations, not only predictions.

This helped us decide to keep the EDA and error analysis central. Even a strong validation score would not be enough if we cannot explain what kinds of categories the model misses.

## <details open><summary>5. How Our Kaggle Task Differs</summary></details>

Our assignment is much smaller than full automated ICD coding:

- The target is a **single first-character category**, not a full ICD code.
- It is **not multi-label**: each literal must receive exactly one `y_category`.
- The inputs are **short literals**, not full EMRs or discharge summaries.
- We do have an ICD description file in the data, but official descriptions are not guaranteed to match the short clinical literals directly.
- The task remains clinically meaningful because the literals are noisy, abbreviated, imbalanced, and sometimes ambiguous.

This helped us decide that long-document strategies from the survey are less urgent, while short-text lexical baselines and Spanish clinical RoBERTa are very relevant.

## <details open><summary>6. Survey Method Map to Our Decisions</summary></details>

The table below translates the survey into our actual project plan. We did not implement every research idea because scope matters: this is a course project and a Kaggle category-prefix task, not a hospital deployment system.

In [ ]:
method_map = pd.read_csv(TABLES / 'survey_method_map.csv')
method_map

### Ideas We Can Actually Implement

After reading the survey, we decided that the feasible part of the project should include:

- a majority baseline,
- TF-IDF character n-grams,
- TF-IDF word n-grams,
- fuzzy matching or nearest-neighbor retrieval if time allows,
- Spanish biomedical-clinical RoBERTa,
- class weighting as an ablation,
- pooling strategies such as CLS vs mean pooling,
- simple ensembling after individual models are validated,
- confidence and error analysis.

This became our implementation path because these methods match the data we observed: short literals, 36 broad categories, strong imbalance, and repeated/ambiguous strings.

### Ideas We Keep as Future Work

We did not implement some survey ideas because they are too large for this assignment or mismatched with the target:

- GNNs over the full ICD hierarchy,
- label-description matching as the central model,
- full ICD code prediction,
- multi-label document-level modeling,
- knowledge graphs,
- clinical deployment and full interpretability workflows.

These became future work because our task is intentionally scoped: one category prefix from one short literal. We can mention them in the report as directions that would matter if the project moved closer to real hospital ICD coding.

## <details open><summary>7. Final Strategy After the Survey</summary></details>

The survey gave us a way to justify the project sequence. After this reading, the order of the repository became part of the method rather than just organization:

1. Start with EDA and annotation design because ICD coding is structured and medically meaningful.
2. Build simple baselines because they are necessary for honest evaluation.
3. Use TF-IDF n-grams because short noisy clinical literals often reward lexical robustness.
4. Use Spanish biomedical-clinical RoBERTa because PLMs are the modern direction and the tokenizer/backbone match our language/domain better than generic English models.
5. Keep hierarchy-aware, graph, and full multi-label methods as future work.

No model is trained in this notebook. It is conceptual grounding and project strategy.

## Follow-up: Information Retrieval as an Implemented Ablation

After reading the survey, we understood that automated ICD coding has often been approached as more than ordinary classification. One older and still intuitive view is information retrieval: given a clinical phrase, retrieve the most similar ICD description or the most similar previously coded example.

We implemented this idea in `models/v03_similarity_retrieval_baseline.py`. Since our repository includes `icd_d_p_pairs.csv`, we tested both nearest training-literal retrieval and literal-to-ICD-description retrieval.

In [ ]:
import pandas as pd
retrieval_grid = pd.read_csv('../reports/tables/v03_similarity_retrieval_grid.csv')
retrieval_grid


## What We Decided After Testing Retrieval

The best retrieval variant was 1-nearest-neighbor over training literals with character TF-IDF `(3,5)`, reaching 0.4974 validation accuracy and 0.4628 macro F1. Direct retrieval against ICD descriptions was weaker.

This helped us decide that retrieval is useful as an ablation and as a way to inspect similar examples, but it should not be our main model. It fails when similar or identical literals have different categories, and it depends heavily on whether the query wording matches the indexed wording. This became one bridge from the survey to our project: we implemented the idea, measured it, and then moved on with evidence.